# Code-based evals, step by step

This workshop's medical-record extraction pipeline is graded two ways:

- **LLM-as-a-Judge** (`evals/eval_hpi_judge.py`, `evals/eval_clinical_note_dataset.py`) -- a model scores subjective quality (accuracy, completeness, tone) on a 0-4 scale.
- **Code-based / deterministic evals** (`evals/eval_diagnoses.py`, `evals/eval_vitals.py`, `evals/eval_trajectory.py`) -- plain Python functions, **no LLM calls at all**. They take a *golden* record (the correct answer) and an *extracted* medical record(what the agent produced), and return a metrics dict: precision/recall, mismatches, fabrications, plausibility, tool-call correctness.

Deterministic evals are cheap, instant, and 100% reproducible -- exactly the checks you want running on every single case, every single run, not just a sampled few via an LLM judge.

**Today:** we'll walk through `eval_diagnoses` together, then you'll build the same kind of check yourself for vitals.


## Setup

`evals/` is a Python package (`evals/__init__.py`), so `eval_diagnoses`, `eval_vitals`, and `eval_trajectory` are importable as `evals.eval_diagnoses`, etc. -- as long as the repo root is on `sys.path` (a script run as `python -m evals.eval_diagnoses` gets this for free; a notebook doesn't, so we add it by hand below).

In [1]:
import sys
import json
from pathlib import Path

In [2]:
repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

In [3]:
from evals.eval_diagnoses import eval_diagnoses, eval_diagnosis_list_limits, load_icd10_catalog_parquet
from evals.eval_vitals import eval_vitals, eval_vitals_plausibility, VITAL_FIELDS, TOLERANCE, PLAUSIBLE_RANGES
from evals.eval_trajectory import eval_trajectory

In [4]:
def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

## The data

- **Golden**: `data/golden/golden_encounter_riv001.json` -- Frodo Baggins, RIV-001, the Weathertop puncture wound case. This is the "correct answer" a human clinician would write.
- **Extracted (mock)**: `data/mocks/extracted_riv001.json` -- a deliberately imperfect extraction we hand-built, so every check below has something real to catch. It's the same shape agent.py's `ClinicalNote` would produce (`vitals`, `differential_diagnoses`, `assessment`).

In [5]:
golden = load_json(repo_root / "data/golden/golden_encounter_riv001.json")
extracted = load_json(repo_root / "data/mocks/extracted_riv001.json")

print("GOLDEN diagnoses:")
print(json.dumps({
    "differential_diagnoses": golden["differential_diagnoses"],
    "assessment": golden["assessment"],
}, indent=2))

print("\nEXTRACTED diagnoses:")
print(json.dumps({
    "differential_diagnoses": extracted["differential_diagnoses"],
    "assessment": extracted["assessment"],
}, indent=2))


GOLDEN diagnoses:
{
  "differential_diagnoses": [
    {
      "icd10_code": "A41.9",
      "diagnosis": "Sepsis, unspecified organism"
    },
    {
      "icd10_code": "T14.8XXA",
      "diagnosis": "Other injury of unspecified body region"
    }
  ],
  "assessment": [
    {
      "icd10_code": "S41.032A",
      "diagnosis": "Puncture wound of left shoulder with signs of progressive infection"
    }
  ]
}

EXTRACTED diagnoses:
{
  "differential_diagnoses": [
    {
      "icd10_code": "A41.9",
      "diagnosis": "Sepsis, unspecified organism"
    },
    {
      "icd10_code": "XX99.99",
      "diagnosis": "Fabricated non-existent code"
    }
  ],
  "assessment": [
    {
      "icd10_code": "S41.032A",
      "diagnosis": "Puncture wound of left shoulder with signs of progressive infection"
    }
  ]
}


Spot the differences before we run any code:
- Golden has **`T14.8XXA`** (other injury) in its differentials, the mock extraction dropped it.
- The mock extraction added **`XX99.99`**: a made-up code that isn't real ICD-10 at all.

That's exactly the kind of thing `eval_diagnoses` is built to catch.


## Diagnoses precision & recall

Precision and recall are computed over the **set of ICD-10 codes** each side uses (differentials + assessment combined):

- **True positive (TP)**: a code that's in *both* golden and extracted.
- **False positive (FP)**: a code the extraction used that golden doesn't have (the agent invented or misapplied it).
- **False negative (FN)**: a code golden has that the extraction missed.

$$\text{precision} = \frac{TP}{TP + FP} \qquad \text{recall} = \frac{TP}{TP + FN}$$

Let's compute it by hand first, then check it against the real `eval_diagnoses` function.


In [6]:
def all_codes(medical_record):
    """Every ICD-10 code across differentials + assessment."""
    codes = set()
    for item in medical_record.get("differential_diagnoses", []) + medical_record.get("assessment", []):
        if item.get("icd10_code"):
            codes.add(item["icd10_code"])
    return codes


golden_codes = all_codes(golden)
extracted_codes = all_codes(extracted)

true_positives = golden_codes & extracted_codes
false_positives = extracted_codes - golden_codes
false_negatives = golden_codes - extracted_codes

precision = len(true_positives) / len(extracted_codes) if extracted_codes else 1.0
recall = len(true_positives) / len(golden_codes) if golden_codes else 1.0

print("golden codes:    ", golden_codes)
print("extracted codes: ", extracted_codes)
print("true positives:  ", true_positives)
print("false positives: ", false_positives, "<- the invented code")
print("false negatives: ", false_negatives, "<- the dropped code")
print()
print(f"precision = {len(true_positives)}/{len(extracted_codes)} = {precision:.2f}")
print(f"recall    = {len(true_positives)}/{len(golden_codes)} = {recall:.2f}")


golden codes:     {'A41.9', 'T14.8XXA', 'S41.032A'}
extracted codes:  {'A41.9', 'XX99.99', 'S41.032A'}
true positives:   {'A41.9', 'S41.032A'}
false positives:  {'XX99.99'} <- the invented code
false negatives:  {'T14.8XXA'} <- the dropped code

precision = 2/3 = 0.67
recall    = 2/3 = 0.67


### Check our work

`eval_diagnoses(golden, extracted, icd10_catalog)` computes the same precision/recall -- plus two extra checks our hand-rolled version doesn't do:

- **`invalid_codes`**: codes not found in the real ICD-10 catalog at all (catches `XX99.99`).
- **`code_diagnosis_mismatches`**: an *optional* LLM-as-judge check (does the diagnosis text actually match what the code means?) -- skipped here since it needs an `anthropic` API key; we're passing no judge callable, so it stays empty.


In [7]:
catalog = load_icd10_catalog_parquet(str(repo_root / "data/ICD10_DB.parquet"))
result = eval_diagnoses(golden, extracted, catalog)

printable = {k: (sorted(v) if isinstance(v, set) else v) for k, v in result.items()}
print(json.dumps(printable, indent=2))

assert result["precision"] == precision
assert result["recall"] == recall
print("\nMatches our hand-computed precision/recall.")


{
  "precision": 0.6666666666666666,
  "recall": 0.6666666666666666,
  "false_positive_codes": [
    "XX99.99"
  ],
  "false_negative_codes": [
    "T14.8XXA"
  ],
  "invalid_codes": [
    "XX99.99"
  ],
  "code_diagnosis_mismatches": []
}

Matches our hand-computed precision/recall.


There's a second, structural check with no golden needed at all: `eval_diagnosis_list_limits` enforces the prompt's own rules (max 3 differentials, max 3 assessment entries, assessment can't be empty) directly against the extraction.


In [ ]:
print(json.dumps(eval_diagnosis_list_limits(extracted), indent=2))

## Your turn: vitals precision & recall

Vitals aren't a list of codes -- they're **5 fixed fields** (`temperature_c`, `heart_rate_bpm`, `respiratory_rate_bpm`, `blood_pressure`, `spo2_percent`), each either present or `null` on both sides. But the *idea* is identical to what we just did for diagnoses:

- **TP**: a field both golden and extracted report (i.e. neither is `null`).
- **FP**: a field extracted reports but golden says isn't knowable (`null` in golden).
- **FN**: a field golden reports but extracted missed (`null` in extracted).

$$\text{presence\_precision} = \frac{TP}{TP+FP} \qquad \text{presence\_recall} = \frac{TP}{TP+FN}$$

Look at the two vitals dicts below, then fill in `my_vitals_precision_recall`.


In [8]:
print("GOLDEN vitals:   ", golden["vitals"])
print("EXTRACTED vitals:", extracted["vitals"])
print("\nAll 5 fields:", VITAL_FIELDS)

GOLDEN vitals:    {'temperature_c': 38.1, 'heart_rate_bpm': 102, 'respiratory_rate_bpm': 20, 'blood_pressure': '100/65', 'spo2_percent': 96}
EXTRACTED vitals: {'temperature_c': 38.1, 'heart_rate_bpm': 130, 'respiratory_rate_bpm': None, 'blood_pressure': '100/65', 'spo2_percent': 900}

All 5 fields: ['temperature_c', 'heart_rate_bpm', 'respiratory_rate_bpm', 'blood_pressure', 'spo2_percent']


In [9]:
def vitals_precision_recall(golden_vitals, extracted_vitals):
    """Return (presence_precision, presence_recall), following the same
    TP/FP/FN pattern as the diagnoses example above."""
    # TODO 1: which fields did the extraction report (i.e. value is not None)?
    reported_by_extraction = None  # <- your code here

    # TODO 2: which fields *should* have been reported (golden's value is not None)?
    should_be_reported = None  # <- your code here

    # TODO 3: true positives = fields in both sets
    true_positives = None  # <- your code here

    # TODO 4: precision = TP / reported_by_extraction, recall = TP / should_be_reported
    #         (both should default to 1.0 if the denominator set is empty)
    precision = None  # <- your code here
    recall = None  # <- your code here

    return precision, recall


vitals_precision_recall(golden["vitals"], extracted["vitals"])


(None, None)

In [10]:
vitals_result = eval_vitals(golden, extracted)
print(json.dumps(vitals_result, indent=2))

answer = vitals_precision_recall(golden["vitals"], extracted["vitals"])
print("\nyour answer:  ", answer)
print("expected:      ", (vitals_result["presence_precision"], vitals_result["presence_recall"]))
assert answer == (vitals_result["presence_precision"], vitals_result["presence_recall"]), \
    "not quite -- check your TP/FP/FN sets against the golden/extracted vitals dicts above"
print("\nCorrect!")


{
  "presence_precision": 1.0,
  "presence_recall": 0.8,
  "value_mismatches": [
    {
      "field": "spo2_percent",
      "golden": 96,
      "extracted": 900
    },
    {
      "field": "heart_rate_bpm",
      "golden": 102,
      "extracted": 130
    }
  ],
  "fabricated_fields": [],
  "missing_fields": [
    "respiratory_rate_bpm"
  ]
}

your answer:   (None, None)
expected:       (1.0, 0.8)


AssertionError: not quite -- check your TP/FP/FN sets against the golden/extracted vitals dicts above

`eval_vitals` also checks two things your precision/recall alone can't see:

- **`value_mismatches`**: fields present on *both* sides but numerically too far apart (tolerance per field -- e.g. ±0.2°C for temperature, ±3bpm for heart rate).
- **`fabricated_fields`**: the dangerous one -- a field the agent invented a number for, where golden says it *isn't knowable at all* (`null`). Kept separate from recall because recall just means "missed something"; fabrication means "made something up".

Both already show up in the `vitals_result` you printed above -- take a look at `value_mismatches` and `missing_fields`.


## Keeping vitals within plausible ranges

Precision/recall/mismatches only compare golden vs. extracted -- they can't catch a value that's just *physiologically impossible*, with no golden needed to know it's wrong. That's `eval_vitals_plausibility`'s job. It checks each field against a generous (extreme-but-survivable) range:


In [11]:
print(json.dumps(PLAUSIBLE_RANGES, indent=2))
print("\nOur mock's spo2_percent:", extracted["vitals"]["spo2_percent"], "  <- is that plausible?")

{
  "temperature_c": [
    30.0,
    43.0
  ],
  "heart_rate_bpm": [
    20,
    250
  ],
  "respiratory_rate_bpm": [
    4,
    60
  ],
  "spo2_percent": [
    50,
    100
  ]
}

Our mock's spo2_percent: 900   <- is that plausible?


In [12]:
def is_within_range(value, low, high):
    """True if value is inside [low, high]. value may be None (nothing to check)."""
    if value is None:
        return True
    return low <= value <= high


def implausible_fields(vitals):
    """Return a list of field names in `vitals` that fall outside PLAUSIBLE_RANGES."""
    implausible = []
    for field, (low, high) in PLAUSIBLE_RANGES.items():
        value = vitals.get(field)
        if value is not None and not is_within_range(value, low, high):
            implausible.append(field)
    return implausible


implausible_fields(extracted["vitals"])

['spo2_percent']

### Confirm it matches `eval_vitals_plausibility`

In [13]:
plausibility_result = eval_vitals_plausibility(extracted)
print(json.dumps(plausibility_result, indent=2))

flagged_fields = {f["field"] for f in plausibility_result["implausible_fields"]}
print("\nyour answer:", implausible_fields(extracted["vitals"]))
print("expected:   ", sorted(flagged_fields))

{
  "implausible_fields": [
    {
      "field": "spo2_percent",
      "value": 900,
      "expected_range": [
        50,
        100
      ]
    }
  ],
  "unparseable_blood_pressure": false
}

your answer: ['spo2_percent']
expected:    ['spo2_percent']


## Trajectory eval

The third deterministic eval, not optional -- it's run on every case alongside diagnoses and vitals in `evals/run_evals.py`. Same TP/FP idea as before, just applied to *tool calls* instead of diagnosis codes or vitals fields. `golden["expected_tool_calls"]` says what tool calls *should* happen; `data/mocks/tool_trace_riv001.json` is a mock trace of what the agent *actually* called.

In [14]:
tool_trace = load_json(repo_root / "data/mocks/tool_trace_riv001.json")

print("expected:", json.dumps(golden["expected_tool_calls"], indent=2))
print("\nactual trace:", json.dumps(tool_trace, indent=2))

expected: [
  {
    "tool": "search_icd10_codes_batch",
    "expected_call_count": 3,
    "expected_query_contains": [
      "puncture",
      "shoulder",
      "sepsis",
      "toxic",
      "wound"
    ]
  }
]

actual trace: [
  {
    "tool": "search_icd10_codes_batch",
    "args": {
      "diagnosis_names": [
        "puncture wound shoulder"
      ]
    }
  },
  {
    "tool": "search_icd10_codes_batch",
    "args": {
      "diagnosis_names": [
        "unrelated headache"
      ]
    }
  }
]


In [15]:
trajectory_result = eval_trajectory(golden, tool_trace)
print(json.dumps(trajectory_result, indent=2))

{
  "tool_call_counts": {
    "search_icd10_codes_batch": 2
  },
  "count_mismatches": [
    "search_icd10_codes_batch: expected 3 call(s), got 2"
  ],
  "argument_mismatches": [
    {
      "tool": "search_icd10_codes_batch",
      "args": {
        "diagnosis_names": [
          "unrelated headache"
        ]
      },
      "expected_keywords": [
        "puncture",
        "shoulder",
        "sepsis",
        "toxic",
        "wound"
      ]
    }
  ],
  "forbidden_query_matches": [],
  "unexpected_tools_called": []
}


Four things caught at once: too few `search icd10 batch` calls, one call whose query didn't mention any of the expected keywords, and an unexpected tool (`get_patient_history`) that was never in the golden's expected trajectory at all.

## Beyond the mocks: pulling real data from Langfuse

Everything above graded a hand-built mock extraction (`data/mocks/extracted_riv001.json`). You don't have to hand-author that fixture forever: once `agent.py` has run once with Langfuse tracing enabled (the `LANGFUSE_*` vars in `.env.example`), every node's real input (the transcript prompt) and real output (the extracted vitals/HPI/diagnoses, plus the ICD-10 tool calls the diagnoses node actually made) already live in Langfuse as trace observations -- no mock needed.

`evals/eval_hpi_judge.py` already does exactly this for the HPI judge (`find_hpi_observation` pulls the `hpi` node's real transcript + real generated HPI off a trace). Below we do the same for the `vitals` and `diagnoses` nodes and the diagnoses node's tool calls, then feed all three straight into the same `eval_vitals` / `eval_diagnoses` / `eval_trajectory` functions used all notebook -- grading a real run instead of a fixture.

Run `uv run python agent.py` at least once first (with `LANGFUSE_PUBLIC_KEY`/`LANGFUSE_SECRET_KEY` set) so a completed trace exists to pull from.

In [17]:
import time

from dotenv import load_dotenv
from langfuse import get_client

from evals.eval_diagnoses import SCORE_CONFIGS as DIAGNOSES_SCORE_CONFIGS, log_diagnoses_scores
from evals.eval_trajectory import SCORE_CONFIGS as TRAJECTORY_SCORE_CONFIGS, log_trajectory_scores
from evals.eval_vitals import SCORE_CONFIGS as VITALS_SCORE_CONFIGS, log_vitals_scores
from evals.judge_client import ensure_score_configs

load_dotenv(repo_root / ".env")
langfuse_client = get_client()

In [18]:
def wait_until(fn, ready, tries=10, delay=1.0):
    """Poll `fn` until it stops raising AND `ready(result)` is True. Local
    Langfuse ingestion is async (SDK -> queue -> langfuse-worker), so right
    after flush() a 404, or a 200 with an empty list, both just mean
    "not indexed yet", not "missing"."""
    last_error = None
    for _ in range(tries):
        try:
            result = fn()
            if ready(result):
                return result
        except Exception as e:
            last_error = e
        time.sleep(delay)
    if last_error:
        raise last_error
    raise TimeoutError("Gave up waiting for Langfuse ingestion")

In [19]:
# Write a small mock trace by hand: a `vitals` span and a `diagnoses` span
# (each holding the same mock `extracted` data used all notebook), plus a
# nested tool-call span under `diagnoses` -- just enough structure to
# exercise the retrieve-and-grade path below. For testing purposes only:
# a real agent.py run would produce this automatically via Langfuse tracing.
with langfuse_client.start_as_current_observation(name="mock-eval-demo", as_type="span") as root_span:
    mock_trace_id = root_span.trace_id

    with langfuse_client.start_as_current_observation(
        name="vitals", as_type="chain", output={"vitals": extracted["vitals"]},
    ):
        pass

    with langfuse_client.start_as_current_observation(
        name="diagnoses",
        as_type="chain",
        output={"diagnoses": {
            "differential_diagnoses": extracted["differential_diagnoses"],
            "assessment": extracted["assessment"],
        }},
    ):
        with langfuse_client.start_as_current_observation(
            name="search_icd10_codes_batch",
            as_type="tool",
            input={"diagnosis_names": ["puncture wound shoulder", "sepsis"]},
            output={"puncture wound shoulder": [{"icd10_code": "S41.032A"}]},
        ):
            pass

langfuse_client.flush()
print("Wrote mock trace:", mock_trace_id)

Wrote mock trace: c8b7c34a0d5770977a5f6a1ac0aa95a4


In [20]:
# Retrieve it back.
trace = wait_until(
    lambda: langfuse_client.api.trace.get(mock_trace_id),
    ready=lambda t: len(t.observations) >= 4,
)
print(f"Retrieved trace {mock_trace_id} with {len(trace.observations)} observations")


def find_node_observation(trace, node_name: str):
    for obs in trace.observations:
        if obs.name == node_name and obs.type == "CHAIN":
            return obs
    raise ValueError(f"No '{node_name}' observation found on trace {trace.id}")


vitals_obs = find_node_observation(trace, "vitals")
diagnoses_obs = find_node_observation(trace, "diagnoses")

retrieved_extracted = {
    "vitals": vitals_obs.output["vitals"],
    "differential_diagnoses": diagnoses_obs.output["diagnoses"]["differential_diagnoses"],
    "assessment": diagnoses_obs.output["diagnoses"]["assessment"],
}
retrieved_tool_trace = [
    {"tool": obs.name, "args": obs.input}
    for obs in trace.observations
    if obs.type == "TOOL"
]
print(json.dumps(retrieved_extracted, indent=2))
print(json.dumps(retrieved_tool_trace, indent=2))

Retrieved trace c8b7c34a0d5770977a5f6a1ac0aa95a4 with 4 observations
{
  "vitals": {
    "temperature_c": 38.1,
    "heart_rate_bpm": 130,
    "respiratory_rate_bpm": null,
    "blood_pressure": "100/65",
    "spo2_percent": 900
  },
  "differential_diagnoses": [
    {
      "icd10_code": "A41.9",
      "diagnosis": "Sepsis, unspecified organism"
    },
    {
      "icd10_code": "XX99.99",
      "diagnosis": "Fabricated non-existent code"
    }
  ],
  "assessment": [
    {
      "icd10_code": "S41.032A",
      "diagnosis": "Puncture wound of left shoulder with signs of progressive infection"
    }
  ]
}
[
  {
    "tool": "search_icd10_codes_batch",
    "args": {
      "diagnosis_names": [
        "puncture wound shoulder",
        "sepsis"
      ]
    }
  }
]


In [21]:
# Same eval functions, same golden, same result as running them on `extracted`
# directly -- this just proves the round trip through Langfuse didn't lose
# or change anything.
vitals_result_rt = eval_vitals(golden, retrieved_extracted)
diagnoses_result_rt = eval_diagnoses(golden, retrieved_extracted, catalog)
trajectory_result_rt = eval_trajectory(golden, retrieved_tool_trace)

print("vitals:     ", vitals_result_rt)
print("diagnoses:  ", {k: (sorted(v) if isinstance(v, set) else v) for k, v in diagnoses_result_rt.items()})
print("trajectory: ", trajectory_result_rt)

vitals:      {'presence_precision': 1.0, 'presence_recall': 0.8, 'value_mismatches': [{'field': 'spo2_percent', 'golden': 96, 'extracted': 900}, {'field': 'heart_rate_bpm', 'golden': 102, 'extracted': 130}], 'fabricated_fields': [], 'missing_fields': ['respiratory_rate_bpm']}
diagnoses:   {'precision': 0.6666666666666666, 'recall': 0.6666666666666666, 'false_positive_codes': ['XX99.99'], 'false_negative_codes': ['T14.8XXA'], 'invalid_codes': ['XX99.99'], 'code_diagnosis_mismatches': []}
trajectory:  {'tool_call_counts': {'search_icd10_codes_batch': 1}, 'count_mismatches': ['search_icd10_codes_batch: expected 3 call(s), got 1'], 'argument_mismatches': [], 'forbidden_query_matches': [], 'unexpected_tools_called': []}


In [22]:
# Log the results back onto the mock trace as Scores -- exactly what
# evals/run_evals.py does for a real run -- then retrieve *those* back too.
all_configs = DIAGNOSES_SCORE_CONFIGS + VITALS_SCORE_CONFIGS + TRAJECTORY_SCORE_CONFIGS
config_ids = ensure_score_configs(langfuse_client, all_configs)

log_vitals_scores(langfuse_client, mock_trace_id, vitals_result_rt, config_ids=config_ids)
log_diagnoses_scores(langfuse_client, mock_trace_id, diagnoses_result_rt, config_ids=config_ids)
log_trajectory_scores(langfuse_client, mock_trace_id, trajectory_result_rt, config_ids=config_ids)
langfuse_client.flush()

scores = wait_until(
    lambda: langfuse_client.api.scores.get_many(trace_id=mock_trace_id),
    ready=lambda s: len(s.data) > 0,
)
print(f"{len(scores.data)} scores now attached to trace {mock_trace_id}:")
for s in scores.data:
    print(f"  {s.name}: {s.value}")

trace c8b7c34a0d5770977a5f6a1ac0aa95a4: 2 vitals value mismatch(es)
trace c8b7c34a0d5770977a5f6a1ac0aa95a4: invalid ICD-10 codes: ['XX99.99']
trace c8b7c34a0d5770977a5f6a1ac0aa95a4: tool call count mismatches: ['search_icd10_codes_batch: expected 3 call(s), got 1']


12 scores now attached to trace c8b7c34a0d5770977a5f6a1ac0aa95a4:
  trajectory_no_forbidden_queries: 1.0
  trajectory_no_unexpected_tools: 1.0
  trajectory_arguments_ok: 1.0
  trajectory_call_counts_match: 0.0
  diagnoses_precision: 0.6666666666666666
  diagnoses_has_code_mismatches: 0.0
  diagnoses_has_invalid_codes: 1.0
  diagnoses_recall: 0.6666666666666666
  vitals_has_fabricated_fields: 0.0
  vitals_has_value_mismatch: 1.0
  vitals_presence_precision: 1.0
  vitals_presence_recall: 0.8
